# Suite JSN — JSON

JSON is a thin, strict (RFC 8259) encoding of Plain data. These cases cover strictness on output and input, text
forms and encodings, value fidelity, and agreement with Plain.

In [ ]:
import json
import math

from schemas.Framework import JSON, Plain, Proxies, Schemas as S
from support import raises, same_graph, text

Item = S.OfObject.Builder().properties(text("s"), text("i", int), text("f", float), text("b", bool),
                                       text("raw", bytes)).create()
Link = S.OfRelation.Builder().links("from", "to").properties(text("w", float)).create()
Item = S.OfObject.Builder(Item).relations(lambda r: r.name("out").of(Link).me("from"),
                                          lambda r: r.name("in").of(Link).me("to")).update()
Proxies.register("Item", Item)
Proxies.register("Link", Link)
B = Proxies.Builders
FromJSON = JSON.FromJSON(B)

## JSN-01 · Output never contains NaN or Infinity literals

In [ ]:
for value in [math.nan, math.inf, -math.inf]:
    out = JSON.ToJSON(Item, B.Item().f(value).create())
    json.loads(out, parse_constant=lambda c: (_ for _ in ()).throw(AssertionError(c)))
    assert json.loads(out)["objects"]["s0"]["f"] in ("NaN", "Infinity", "-Infinity")
with raises(ValueError):
    JSON.dumps({"x": math.nan})  # plain data should never hold one, and dumps refuses it

## JSN-02 · Input: non-standard constants, duplicate keys and syntax errors are rejected

In [ ]:
for bad in ['{"a": NaN}', '{"a": Infinity}', '{"a": -Infinity}', '[NaN]',
            '{"a": 1, "a": 2}', '{"x": {"a": 1, "a": 1}}', '[{"k": 1, "k": 1}]',
            '{"a": 1,}', "{'a': 1}", '{"a": 1} // c', '{"a": 01}', '', '   ', '{"a": 1} {"b": 2}', '{"a": .5}']:
    with raises(ValueError):
        JSON.loads(bad)

## JSN-03 · Text forms: str, UTF-8/16/32 bytes, and byte-order marks

In [ ]:
doc = '{"a": "é"}'
assert JSON.loads(doc) == JSON.loads(doc.encode("utf-8")) == {"a": "é"}
for encoding in ["utf-16", "utf-16-le", "utf-16-be", "utf-32", "utf-8-sig"]:
    assert JSON.loads(doc.encode(encoding)) == {"a": "é"}, encoding
with raises(ValueError, match="BOM"):
    JSON.loads("﻿" + doc)  # a BOM inside a str is an error; in bytes it is accepted
with raises(ValueError):
    JSON.loads(b"\xff\xfe\x00")  # undecodable bytes

## JSN-04 · Output format: key order kept, non-ASCII kept, indentation optional

In [ ]:
item = B.Item().raw(b"\x01").s("日本").i(1).create()
compact = JSON.ToJSON(Item, item)
assert compact == '{"root": "s0", "objects": {"s0": {"s": "日本", "i": 1, "raw": "AQ=="}}}'
pretty = JSON.ToJSON(Item, item, indent=2)
assert "\n" in pretty and JSON.loads(pretty) == JSON.loads(compact)
assert JSON.ToJSON.OfObject(Item, item, indent=4) == JSON.dumps(Plain.ToPlain(Item, item), indent=4)

## JSN-05 · Value fidelity: integers of any size, exact floats, every string

In [ ]:
for value in [0, -1, 2**53 + 1, 2**100, -(2**200)]:
    assert FromJSON(Item, JSON.ToJSON(Item, B.Item().i(value).create())).i == value
for value in [0.1, 1 / 3, 1e308, 5e-324, 2.2250738585072014e-308, -0.0, 123456789.123456789]:
    back = FromJSON(Item, JSON.ToJSON(Item, B.Item().f(value).create())).f
    assert back == value and math.copysign(1, back) == math.copysign(1, value)
for value in ["", "\x00", "\x1f\x7f", '"\\/', "  ", "\ud800", "\U0001f600", "a\r\nb"]:
    assert FromJSON(Item, JSON.ToJSON(Item, B.Item().s(value).create())).s == value, repr(value)
assert FromJSON(Item, JSON.ToJSON(Item, B.Item().raw(bytes(range(256))).create())).raw == bytes(range(256))

## JSN-06 · A lone surrogate is valid Python JSON text but cannot be encoded to UTF-8 (pinned)

In [ ]:
lone = JSON.ToJSON(Item, B.Item().s("\ud800").create())
with raises(UnicodeEncodeError):
    lone.encode("utf-8")
assert "\ud800" in lone  # callers writing files should use errors="surrogatepass" or reject such strings

## JSN-07 · Natives at the top level

In [ ]:
assert JSON.ToJSON(S.OfNative.Data(int), 5) == "5"
assert JSON.ToJSON(S.OfNative.Data(bytes), b"\xff") == '"/w=="'
assert JSON.ToJSON.OfNative(S.OfNative.Data(float), math.inf) == '"Infinity"'
assert FromJSON(S.OfNative.Data(str), '"x"') == FromJSON.OfNative(S.OfNative.Data(str), '"x"') == "x"
with raises(TypeError):
    FromJSON(S.OfNative.Data(int), "true")
with raises(TypeError):
    FromJSON(S.OfNative.Data(float), "1")  # JSON 1 is an integer, not a float

## JSN-08 · JSON agrees with Plain, and graphs round trip

In [ ]:
hub = B.Item().s("hub").out(lambda x: x.to(lambda y: y.s("leaf1")).w(1.0)).out(
    lambda x: x.to(lambda y: y.s("leaf2")).w(math.nan)).create()
graph = Plain.ToPlain.Reachable(Item, hub)
text_ = JSON.ToJSON.Reachable(Item, hub)
assert JSON.loads(text_) == graph
back = FromJSON.Reachable(Item, text_)
assert back is not hub and same_graph(Plain.ToPlain.Reachable(Item, back), graph)
with raises(ValueError, match="unresolved reference"):
    FromJSON(Item, JSON.ToJSON(Item, hub))
with raises(ValueError):
    FromJSON.Reachable(Item, text_.replace('"s1"', '"s9"', 1))

## JSN-09 · Parse errors match Python's `json` messages exactly
The table was generated from Python's `json` and is shared verbatim with the TypeScript suite, whose parser is its
own. Rows: input (text, or bytes as a list), expected message (or None when it parses), whether the message must match
exactly (codec errors from UTF-8/UTF-16 decoding match by prefix), and the repr of the parsed value.

In [ ]:
import json as stdlib_json

TABLE = stdlib_json.loads(r'''[["", "Expecting value: line 1 column 1 (char 0)", true, null], ["   ", "Expecting value: line 1 column 4 (char 3)", true, null], ["-", "Expecting value: line 1 column 1 (char 0)", true, null], ["nul", "Expecting value: line 1 column 1 (char 0)", true, null], ["tru", "Expecting value: line 1 column 1 (char 0)", true, null], [".5", "Expecting value: line 1 column 1 (char 0)", true, null], ["[1,]", "Illegal trailing comma before end of array: line 1 column 3 (char 2)", true, null], ["{\"a\": 1,}", "Illegal trailing comma before end of object: line 1 column 8 (char 7)", true, null], ["{\"a\" 1}", "Expecting ':' delimiter: line 1 column 6 (char 5)", true, null], ["{1: 2}", "Expecting property name enclosed in double quotes: line 1 column 2 (char 1)", true, null], ["{\"a\": 1 \"b\": 2}", "Expecting ',' delimiter: line 1 column 9 (char 8)", true, null], ["[1 2]", "Expecting ',' delimiter: line 1 column 4 (char 3)", true, null], ["\"abc", "Unterminated string starting at: line 1 column 1 (char 0)", true, null], ["\"a\\", "Unterminated string starting at: line 1 column 1 (char 0)", true, null], ["\"a\\x\"", "Invalid \\escape: line 1 column 3 (char 2)", true, null], ["\"\\u12\"", "Invalid \\uXXXX escape: line 1 column 3 (char 2)", true, null], ["\"\\uZZZZ\"", "Invalid \\uXXXX escape: line 1 column 3 (char 2)", true, null], ["\"a\nb\"", "Invalid control character at: line 1 column 3 (char 2)", true, null], ["\"a\tb\"", "Invalid control character at: line 1 column 3 (char 2)", true, null], ["1 2", "Extra data: line 1 column 3 (char 2)", true, null], ["{\"a\": 1} x", "Extra data: line 1 column 10 (char 9)", true, null], ["[", "Expecting value: line 1 column 2 (char 1)", true, null], ["{", "Expecting property name enclosed in double quotes: line 1 column 2 (char 1)", true, null], ["{\"a\"", "Expecting ':' delimiter: line 1 column 5 (char 4)", true, null], ["{\"a\":", "Expecting value: line 1 column 6 (char 5)", true, null], ["01", "Extra data: line 1 column 2 (char 1)", true, null], ["1.", "Extra data: line 1 column 2 (char 1)", true, null], ["1e", "Extra data: line 1 column 2 (char 1)", true, null], ["[\n  1,\n  x]", "Expecting value: line 3 column 3 (char 9)", true, null], ["{\n\"a\": [\n1,\n2\n]\n,}", "Illegal trailing comma before end of object: line 6 column 1 (char 16)", true, null], ["NaN", "NaN is not valid JSON", true, null], ["-Infinity", "-Infinity is not valid JSON", true, null], ["[Infinity]", "Infinity is not valid JSON", true, null], ["99999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999999", "Exceeds the limit (4300 digits) for integer string conversion: value has 4301 digits", true, null], [[0, 0, 0, 49], null, true, "1"], [[49, 0, 0, 0], null, true, "1"], [[254, 255, 0, 49], null, true, "1"], [[0, 49], null, true, "1"], [[49, 0], null, true, "1"], [[0, 0, 254, 255, 0, 0, 0, 49], null, true, "1"], [[0, 0, 0], "Expecting value: line 1 column 1 (char 0)", true, null], [[255, 254, 49], "'utf-16-le' codec can't decode", false, null], [[195, 40], "'utf-8' codec can't decode", false, null], [[255, 254, 0, 0, 0, 0, 17, 0], "'utf-32-le' codec can't decode bytes in position 4-7: code point not in range(0x110000)", true, null], [[0, 0, 0, 49, 50], "'utf-32-be' codec can't decode byte 0x32 in position 4: truncated data", true, null], [[0, 0, 0, 49, 50, 51], "'utf-32-be' codec can't decode bytes in position 4-5: truncated data", true, null], [[239, 187, 191, 49], null, true, "1"]]''')
assert len(TABLE) == 47
for source, message, exact, result in TABLE:
    data = bytes(source) if isinstance(source, list) else source
    try:
        value = JSON.loads(data)
    except ValueError as error:
        assert message is not None, (source, str(error))
        assert (str(error) == message) if exact else str(error).startswith(message), (source, str(error))
    else:
        assert message is None and repr(value) == result, (source, value)
assert JSON.loads('["\\ud83d\\ude00", "\\ud800", "\\ud800x", "\\udc00", "\\b\\f\\n\\r\\t\\/\\\\\\""]') == [
    "\U0001f600", "\ud800", "\ud800x", "\udc00", "\b\f\n\r\t/\\\""]

## JSN-10 · Only plain data can be written
Python's `json` would turn an int, float, bool or None key into a string; plain data only has string keys.

In [ ]:
with raises(TypeError, match="keys must be str, not int"):
    JSON.dumps({1: 2})
with raises(TypeError, match="keys must be str, not int"):
    JSON.dumps({"a": [{"b": {1: 2}}]})
for value, name in [({1, 2}, "set"), (b"x", "bytes"), (object(), "object")]:
    with raises(TypeError, match=f"Object of type {name} is not JSON serializable"):
        JSON.dumps({"a": [value]})

## JSN-11 · Every kind of plain value is written and read, in Python's exact text

In [ ]:
value = [None, True, False, 0, -1, 1.5, "s", [], {}, {"a": [None, {}]}, [[]]]
assert JSON.dumps(value) == '[null, true, false, 0, -1, 1.5, "s", [], {}, {"a": [null, {}]}, [[]]]'
assert JSON.dumps(value, indent=1) == '[\n null,\n true,\n false,\n 0,\n -1,\n 1.5,\n "s",\n [],\n {},\n {\n  "a": [\n   null,\n   {}\n  ]\n },\n [\n  []\n ]\n]'
assert JSON.loads(JSON.dumps(value)) == value and JSON.loads(JSON.dumps(value, indent=1)) == value
assert JSON.loads(" null ") is None and JSON.loads("{}") == {} and JSON.loads("[ ]") == [] and JSON.loads('{ }') == {}